In [114]:
# Data configuration
SEED = 42
DATA_ID = 'TIGER-Lab/ViRL39K'
DATA_SPLIT = 'train'
DATA_SIZE = 1250
TEST_RATIO = 0.2

In [115]:
from datasets import load_dataset, Dataset
from collections import deque

def load_hf_dataset(data_id, split, total_size, from_end=False):
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    if not from_end:
        # Slice from the start
        sliced_data = []
        for example in dataset_stream:
            if len(sliced_data) >= total_size:
                break
            sliced_data.append(example)
    else:
        # Slice from the end using a fixed-size buffer
        buffer = deque(maxlen=total_size)
        for example in dataset_stream:
            buffer.append(example)
        sliced_data = list(buffer)

    dataset = Dataset.from_list(sliced_data)
    return dataset

# Load the RL dataset
dataset = load_hf_dataset(
    DATA_ID, 
    split=DATA_SPLIT,
    total_size=38_869, # Actual size is 38,870, but loading all causes an error, so we load one less than the actual size
    from_end=False,
)

# Add 'category_label' column and encode it
dataset = dataset.add_column('category_label', dataset['category'])
dataset = dataset.class_encode_column('category_label')

print(dataset)

Casting to class labels:   0%|          | 0/38869 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image', 'category_label'],
    num_rows: 38869
})


In [116]:
# Filter out examples with multiple images
dataset_filtered = dataset.filter(lambda x: len(x['image']) == 1)
print(dataset_filtered)

Filter:   0%|          | 0/38869 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image', 'category_label'],
    num_rows: 36580
})


In [117]:
import numpy as np
from datasets import concatenate_datasets

# Get normalized distribution
category_counts = dataset.to_pandas()['category_label'].value_counts(normalize=True)

# Raw allocation (float)
raw = category_counts * DATA_SIZE

# Floor allocation
samples_per_category = np.floor(raw).astype(int)

# Distribute remainder
remainder = DATA_SIZE - samples_per_category.sum()
fractional = raw - samples_per_category

for cat in fractional.sort_values(ascending=False).index[:remainder]:
    samples_per_category[cat] += 1

# Stratified sampling
sampled_splits = []
for category, n in samples_per_category.items():
    subset = dataset_filtered.filter(lambda x: x['category_label'] == category)
    subset = subset.shuffle(seed=SEED).select(range(min(n, len(subset))))
    sampled_splits.append(subset)

# Combine all
dataset_sampled = concatenate_datasets(sampled_splits)
print(dataset_sampled)

Filter:   0%|          | 0/36580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36580 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image', 'category_label'],
    num_rows: 1250
})


In [118]:
# Download the images.zip file
from huggingface_hub import hf_hub_download

images_zip_path = hf_hub_download(
    repo_id=DATA_ID,
    filename='images.zip',
    repo_type='dataset',
)
print(images_zip_path)

/root/.cache/huggingface/hub/datasets--TIGER-Lab--ViRL39K/snapshots/812ec617dea4bc8a4e751663b88e4ebb7de4d00e/images.zip


In [119]:
# Extract the images.zip file
import os
if not os.path.isdir('images'):
    import zipfile
    with zipfile.ZipFile(images_zip_path, 'r') as zip_ref:
        zip_ref.extractall()

In [158]:
# Check category distribution in the processed dataset
df = dataset_sampled.to_pandas()
cat_counts = df[['category', 'category_label']].value_counts()
cat_counts

,,count
category,category_label,
(GradeSchool) Non-Geo Math,1,479
(GradeSchool) Geometric,0,342
Tables/Diagrams/Charts,7,199
(GradeSchool) Science,2,111
Spatial Reasoning,6,62
Broader STEM Topics,3,31
Social Science,5,22
Commonsense,4,4


In [159]:
# Exclude samples from the smallest category before stratification
smallest_cat_count = cat_counts.iloc[-1]
smallest_cat_label = cat_counts.index[-1][1]

drop_idxs = df[df['category_label'] == smallest_cat_label].sample(smallest_cat_count-3, random_state=SEED).index
to_stratify_df = df.drop(index=drop_idxs)
no_stratify_df = df.loc[drop_idxs]

print("Number of samples to stratify:", len(to_stratify_df))
print("Number of samples not to stratify:", len(no_stratify_df))

Number of samples to stratify: 1249
Number of samples not to stratify: 1


In [160]:
# Stratified split the dataset into train, validation, and test sets
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import DatasetDict

train_df, val_test_df = train_test_split(
    to_stratify_df,
    test_size=TEST_RATIO,
    stratify=to_stratify_df['category_label'],
    random_state=SEED,
)
val_test_df = pd.concat([val_test_df, no_stratify_df], ignore_index=True)
val_df, test_df = train_test_split(
    val_test_df,
    test_size=0.5,
    stratify=val_test_df['category_label'],
    random_state=SEED,
)

print("Train set size:", len(train_df))
print("Validation set size:", len(val_df))
print("Test set size:", len(test_df))

Train set size: 999
Validation set size: 125
Test set size: 126


In [161]:
# Manually fix the split set sizes
test_df_sampled = test_df[test_df['category_label'] == 0].sample(1, random_state=SEED)
train_df = pd.concat([train_df, test_df_sampled], ignore_index=True)
test_df = test_df.drop(index=test_df_sampled.index)

print("Train set size:", len(train_df))
print("Validation set size:", len(val_df))
print("Test set size:", len(test_df))

Train set size: 1000
Validation set size: 125
Test set size: 125


In [162]:
# Drop the 'category_label' column from the final splits
train_df = train_df.drop(columns=['category_label'])
val_df = val_df.drop(columns=['category_label'])
test_df = test_df.drop(columns=['category_label'])

train_set = Dataset.from_pandas(train_df, preserve_index=False)
val_set = Dataset.from_pandas(val_df, preserve_index=False)
test_set = Dataset.from_pandas(test_df, preserve_index=False)

print("Train set:")
print(train_set)
print()
print("Validation set:")
print(val_set)
print()
print("Test set:")
print(test_set)

Train set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image'],
    num_rows: 125
})

Test set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image'],
    num_rows: 125
})


In [ ]:
from PIL import Image

def process_image_with_path(example):
    image_paths = example['image']
    assert len(image_paths) == 1, "Expected exactly one image path per example!"
    image_path = image_paths[0]
    image = Image.open(image_path)
    example['PIL_image'] = image
    return example

# Process the split sets to load images
train_set = train_set.map(process_image_with_path, num_proc=4)
val_set = val_set.map(process_image_with_path, num_proc=4)
test_set = test_set.map(process_image_with_path, num_proc=4)

# Rename columns for clarity
rename_cols_map = {'image': 'image_paths', 'PIL_image': 'image'}
train_set = train_set.rename_columns(rename_cols_map)
val_set = val_set.rename_columns(rename_cols_map)
test_set = test_set.rename_columns(rename_cols_map)

assert (
    isinstance(train_set[0]['image'], Image.Image) and
    isinstance(val_set[0]['image'], Image.Image) and
    isinstance(test_set[0]['image'], Image.Image)
), "Expected 'image' column to contain PIL Image objects!"

print("Train set:")
print(train_set)
print()
print("Validation set:")
print(val_set)
print()
print("Test set:")
print(test_set)

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/125 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/125 [00:00<?, ? examples/s]

Train set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 125
})

Test set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 125
})


In [164]:
# Create a DatasetDict for the splits
dataset_split = DatasetDict({
    'train': train_set,
    'validation': val_set,
    'test': test_set,
})
print(dataset_split)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 125
    })
    test: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 125
    })
})


In [166]:
# Push the dataset to Hugging Face Hub
dataset_split.push_to_hub(f'alxxtexxr/ViRL{DATA_SIZE/1000}K')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  59%|#####9    | 3.94MB / 6.68MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/ViRL1.25K/commit/7433d7f6db70062141a9996374affca98720b66c', commit_message='Upload dataset', commit_description='', oid='7433d7f6db70062141a9996374affca98720b66c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/ViRL1.25K', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/ViRL1.25K'), pr_revision=None, pr_num=None)